# Implementation on the CPU-based system

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$This notebook explains how the CPU timing estimates for the classical MCMC algorithm are obtained.

**Table of contents**
1. Source
2. Algorithm
3. How to calculate the timing
4. Reproducibility

## Source

Both the code and the data are available in the `estimation_timing/cpu` folder.

The folder contains:

* `timing_estimation_cpu.cpp`: C++ source code for the CPU benchmark. It implements both the local single-spin-flip Metropolis kernel and the uniform dense-proposal kernel.
* `timing_estimation_cpu.slurm`: SLURM launch script used to compile and run the CPU benchmark on the cluster for the selected values of $n$.
* `log_timing_estimation_cpu_native_20_100k.out`: output log for the native compilation setting, using `-march=native -mtune=native`, with `N_MODELS = 10` and `N_STEPS = 100000`.
* `log_timing_estimation_cpu_sapphire_20_100k.out`: output log for the Sapphire-Rapids-targeted compilation setting, using `-march=sapphirerapids -mprefer-vector-width=512`, with `N_MODELS = 10` and `N_STEPS = 100000`.

## Algorithm

The C++ program benchmarks one Metropolis update for two proposal kernels. Given a current state $x$ and a proposed state $y$, the move is accepted according to the usual Metropolis rule, with acceptance probability $\min\{1,\exp[-\beta(E(y)-E(x))]\}$.

**Note**: the timing estimate does *not* include the time required to generate pseudo-random numbers. The benchmark generates all random data before the timed region, so the measured time reflects the arithmetic kernels rather than pseudo-random number generation.

The implementation uses single-precision `float` values for the fields, couplings, energies, and Metropolis probabilities. For this benchmark, it is more convenient to use the highly optimized floating-point implementation than to introduce a custom fixed-point representation.

The dense SK instance is stored in the `IsingModel` structure. The local fields are stored in a length-$n$ array, while the couplings are stored as a full row-major $n\times n$ matrix. This uses more memory than storing only the upper triangular part, but it makes the dense row access in the local kernel cheaper.

The spin state is stored as an unpacked array with entries in $\{-1,+1\}$. This also uses more memory than a bitstring, but it avoids bit extraction inside the dense arithmetic loops.

The `RandPool` helper class precomputes random integers and uniform random floats. The same random pools are reset before each timed kernel, so the benchmark does not measure random-number generation time.

### CPU local implementation

The local CPU kernel proposes a single spin flip and computes the corresponding energy difference using $\Delta_i E = 2s_i(h_i+\sum_j J_{ij}s_j)$. This costs $O(n)$ arithmetic per proposal, because the update only needs the dense local field of the selected spin.

### CPU uniform implementation

The uniform CPU kernel samples a complete proposed spin configuration and recomputes its full dense SK energy. The dense energy is $E(y) = -\sum_i h_i y_i - \sum_{i<j} J_{ij} y_i y_j$, so each proposal costs $O(n^2)$ arithmetic.

## How to calculate the timing

We ran chains for systems of size $n = 64, 128, 256, 512$ on the Leonardo DCGP partition. The log files report an Intel Xeon Platinum 8358 CPU at 2.60 GHz. The benchmark was run twice with different compiler settings: a native build and a Sapphire-Rapids-targeted build. The attached SLURM script shows the native build; the Sapphire-Rapids-targeted log corresponds to the same script with the `CXXFLAGS` line changed accordingly.

The following Python code fits the measured total runtimes. The local timing is fit as $a+bn$, while the uniform timing is fit as $A+Bn+Cn^2$. The per-operation coefficients are obtained by dividing by `N_MODELS * N_STEPS`.

In [1]:
import numpy as np
from scipy.optimize import nnls

N_MODELS = 10
N_STEPS = 100000
N_OPS = N_MODELS * N_STEPS

ns = np.array([64, 128, 256, 512], dtype=float)

# From log_timing_estimation_cpu_native_20_100k.out.
local_time_native = np.array([0.0189299, 0.0254903, 0.0485925, 0.0957941])
uniform_time_native = np.array([0.77873, 1.97, 6.40592, 25.629])

# From log_timing_estimation_cpu_sapphire_20_100k.out.
local_time_sapphire = np.array([0.0183836, 0.0221172, 0.0399847, 0.0805035])
uniform_time_sapphire = np.array([0.785426, 2.81768, 7.54626, 24.2614])

def fit(y: np.ndarray, deg: int) -> np.ndarray:
    """Return the non-negative least-squares polynomial fit coefficients."""
    return nnls(np.vstack([ns**k for k in range(deg + 1)]).T, y)[0]

for name, local, uniform in [("native", local_time_native, uniform_time_native), ("sapphire", local_time_sapphire, uniform_time_sapphire)]:
    a, b = fit(local, 1)
    A, B, C = fit(uniform, 2)

    print("Compilation settings:", name)
    print(f"  local:     {a:.3e} + {b:.3e} n")
    print(f"  uniform:   {A:.3e} + {B:.3e} n + {C:.3e} n^2")
    print(f"  local/op:  {a/N_OPS:.3e} + {b/N_OPS:.3e} n")
    print(f"  uniform/op:{A/N_OPS:.3e} + {B/N_OPS:.3e} n + {C/N_OPS:.3e} n^2\n")

Compilation settings: native
  local:     5.122e-03 + 1.753e-04 n
  uniform:   3.024e-01 + 0.000e+00 n + 9.643e-05 n^2
  local/op:  5.122e-09 + 1.753e-10 n
  uniform/op:3.024e-07 + 0.000e+00 n + 9.643e-11 n^2

Compilation settings: sapphire
  local:     5.959e-03 + 1.429e-04 n
  uniform:   0.000e+00 + 1.173e-02 n + 6.964e-05 n^2
  local/op:  5.959e-09 + 1.429e-10 n
  uniform/op:0.000e+00 + 1.173e-08 n + 6.964e-11 n^2



Using the recorded timings, this gives approximately:

- native local total time: $5.12\times 10^{-3} + 1.75\times 10^{-4} n$ seconds;
- native uniform total time: $3.02\times 10^{-1} + 9.64\times 10^{-5} n^2$ seconds;
- Sapphire-Rapids-targeted local total time: $5.96\times 10^{-3} + 1.43\times 10^{-4} n$ seconds;
- Sapphire-Rapids-targeted uniform total time: $1.17\times 10^{-2}n + 6.96\times 10^{-5}n^2$ seconds.

After division by `N_OPS = 10 * 100000`, the per-proposal native fits are approximately:

- local: $5.12\times 10^{-9} + 1.75\times 10^{-10} n$ seconds;
- uniform: $3.02\times 10^{-7} + 9.64\times 10^{-11} n^2$ seconds.

The Sapphire-Rapids-targeted build gives comparable results. It improves the local timing slightly and changes the fitted decomposition of the uniform timing, but it does not qualitatively change the scaling. This is consistent with the structure of the benchmark: dense arithmetic can benefit from vector instructions, but the full Metropolis proposal also contains data-dependent control flow and conditional exponential evaluation.

## Reproducibility

To reproduce the CPU timing benchmark, compile `timing_estimation_cpu.cpp` once per system size, passing the system size as a compile-time macro. The tested sizes were $n\in\{64,128,256,512\}$. The recorded log files correspond to runs with `N_MODELS = 10` independent models and `N_STEPS = 100000` Metropolis proposals per model.

The native compilation used GCC with aggressive optimization flags:

    -std=c++20 -O3 -march=native -mtune=native -ffast-math -fno-math-errno -funroll-loops -DNDEBUG

The Sapphire-Rapids-targeted build used:

    -std=c++20 -O3 -march=sapphirerapids -mprefer-vector-width=512 -ffast-math -fno-math-errno -funroll-loops -DNDEBUG

The relevant flags mean:

- `-O3`: enables aggressive compiler optimizations.
- `-march=native` and `-mtune=native`: specialize code generation and scheduling to the host CPU.
- `-march=sapphirerapids`: explicitly targets the Intel Sapphire Rapids instruction set.
- `-mprefer-vector-width=512`: asks GCC to prefer 512-bit vector operations when profitable.
- `-ffast-math`: allows non-strict floating-point optimizations.
- `-fno-math-errno`: tells the compiler that the program does not inspect `errno` after math-library calls.
- `-funroll-loops`: allows profitable loop unrolling.
- `-DNDEBUG`: disables debug assertions.

Run the benchmark through the provided SLURM script:

    sbatch timing_estimation_cpu.slurm

The benchmark should generate logs analogous to:

    log_timing_estimation_cpu_native_20_100k.out
    log_timing_estimation_cpu_sapphire_20_100k.out

If `N_MODELS`, `N_STEPS`, or the tested system sizes change in the C++ source or SLURM script, update the fitting code accordingly.